# Cortex Agents: Role-Based Access Control (RBAC) Guide

**Purpose**: Examples and templates for implementing RBAC for Cortex Agents, including agent creation, tool configuration, and privilege management.

**Last Updated**: March 2026

**Version**: 2.0

## Overview

Cortex Agents orchestrate across Cortex Analyst (semantic views), Cortex Search services, and custom tools. This runbook covers account-level access controls, schema/agent-level privileges, and granting access to underlying tools.

### Prerequisites

- `ACCOUNTADMIN` role
- Existing Cortex Search service (from Notebook 02)
- Existing Semantic View (from Notebook 03)
- Warehouse for query execution

### Key Concepts

**Agent Privileges**: `CREATE AGENT`, `USAGE`, `MODIFY`, `OWNERSHIP`, `MONITOR`

**Database Roles**: `SNOWFLAKE.CORTEX_AGENT_USER` or `SNOWFLAKE.CORTEX_USER`

**Tool Requirements**: Agents require privileges on their underlying tools (search services, semantic views). The agent creator's role must have access to all referenced tools at creation time.

## Part 1: Configuration Variables

In [ ]:
USE SECONDARY ROLES NONE;

SET DATABASE_NAME = 'RUNBOOKS_DB';
SET SCHEMA_NAME = 'CORTEX_AGENTS';
SET WAREHOUSE_NAME = 'COMPUTE_WH';

SET FULL_SCHEMA_PATH = $DATABASE_NAME || '.' || $SCHEMA_NAME;

SET AGENT_CREATOR_ROLE = 'AGENT_CREATOR_ROLE';
SET AGENT_USER_ROLE = 'AGENT_USER_ROLE';
SET AGENT_OPERATOR_ROLE = 'AGENT_OPERATOR_ROLE';

SET AGENT_NAME = 'SALES_ANALYTICS_AGENT';

-- ============================================================================
-- PATHS TO EXISTING TOOLS FROM PREVIOUS NOTEBOOKS
-- ============================================================================
-- Cortex Search Service (from 02_CortexSearch notebook)
SET SEARCH_SERVICE_FULL_PATH = 'RUNBOOKS_DB.CORTEX_SEARCH.PRODUCT_SEARCH_SERVICE';

-- Semantic View (from 03_CortexAnalyst notebook)
SET SEMANTIC_VIEW_FULL_PATH = 'RUNBOOKS_DB.CORTEX_ANALYST.SALES_SEMANTIC_VIEW_RESTRICTED';

-- Warehouse used by tools (from previous notebooks)
SET TOOLS_WAREHOUSE = 'COMPUTE_WH';
-- ============================================================================

-- Display configuration
SELECT 
    $DATABASE_NAME AS DATABASE_NAME,
    $SCHEMA_NAME AS SCHEMA_NAME,
    $FULL_SCHEMA_PATH AS FULL_SCHEMA_PATH,
    $AGENT_NAME AS AGENT_NAME,
    $SEARCH_SERVICE_FULL_PATH AS SEARCH_SERVICE_PATH,
    $SEMANTIC_VIEW_FULL_PATH AS SEMANTIC_VIEW_PATH,
    $TOOLS_WAREHOUSE AS TOOLS_WAREHOUSE;

## Part 2: RBAC Setup

Create custom roles and establish privilege hierarchy.

### 2.1: Create Custom Roles

In [ ]:
USE ROLE ACCOUNTADMIN;

CREATE ROLE IF NOT EXISTS IDENTIFIER($AGENT_CREATOR_ROLE) 
    COMMENT = 'Role for creating and managing Cortex Agents';

CREATE ROLE IF NOT EXISTS IDENTIFIER($AGENT_USER_ROLE) 
    COMMENT = 'Role for querying/using Cortex Agents';

CREATE ROLE IF NOT EXISTS IDENTIFIER($AGENT_OPERATOR_ROLE) 
    COMMENT = 'Role for modifying agent configurations';

### 2.2: Establish Role Hierarchy

In [ ]:
GRANT ROLE IDENTIFIER($AGENT_USER_ROLE) TO ROLE IDENTIFIER($AGENT_OPERATOR_ROLE);
GRANT ROLE IDENTIFIER($AGENT_OPERATOR_ROLE) TO ROLE IDENTIFIER($AGENT_CREATOR_ROLE);

GRANT ROLE IDENTIFIER($AGENT_CREATOR_ROLE) TO ROLE SYSADMIN;

### 2.3: Grant Warehouse Access

In [ ]:
CREATE WAREHOUSE IF NOT EXISTS IDENTIFIER($WAREHOUSE_NAME)
    WITH WAREHOUSE_SIZE = 'XSMALL'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE;

GRANT USAGE ON WAREHOUSE IDENTIFIER($WAREHOUSE_NAME) TO ROLE IDENTIFIER($AGENT_CREATOR_ROLE);
GRANT USAGE ON WAREHOUSE IDENTIFIER($WAREHOUSE_NAME) TO ROLE IDENTIFIER($AGENT_USER_ROLE);
GRANT USAGE ON WAREHOUSE IDENTIFIER($WAREHOUSE_NAME) TO ROLE IDENTIFIER($AGENT_OPERATOR_ROLE);

-- Also grant access to the tools warehouse
GRANT USAGE ON WAREHOUSE IDENTIFIER($TOOLS_WAREHOUSE) TO ROLE IDENTIFIER($AGENT_CREATOR_ROLE);
GRANT USAGE ON WAREHOUSE IDENTIFIER($TOOLS_WAREHOUSE) TO ROLE IDENTIFIER($AGENT_USER_ROLE);

## Part 3: Account-Level Access Controls

Configure account-level database roles for Cortex Agents access.

### 3.1: Grant SNOWFLAKE.CORTEX_AGENT_USER Database Role

This database role provides access specifically to Cortex Agents functionality.

In [ ]:
USE ROLE ACCOUNTADMIN;

GRANT DATABASE ROLE SNOWFLAKE.CORTEX_AGENT_USER TO ROLE IDENTIFIER($AGENT_CREATOR_ROLE); 
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_AGENT_USER TO ROLE IDENTIFIER($AGENT_USER_ROLE);
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_AGENT_USER TO ROLE IDENTIFIER($AGENT_OPERATOR_ROLE);

## Part 4: Create Database and Schema

In [ ]:
USE ROLE ACCOUNTADMIN;

CREATE DATABASE IF NOT EXISTS IDENTIFIER($DATABASE_NAME);
CREATE SCHEMA IF NOT EXISTS IDENTIFIER($FULL_SCHEMA_PATH);

GRANT USAGE ON DATABASE IDENTIFIER($DATABASE_NAME) TO ROLE IDENTIFIER($AGENT_CREATOR_ROLE);
GRANT USAGE ON SCHEMA IDENTIFIER($FULL_SCHEMA_PATH) TO ROLE IDENTIFIER($AGENT_CREATOR_ROLE);

GRANT CREATE AGENT ON SCHEMA IDENTIFIER($FULL_SCHEMA_PATH) TO ROLE IDENTIFIER($AGENT_CREATOR_ROLE);

## Part 5: Grant Access to Existing Tools

**Instead of creating new tools**, we'll grant access to the existing Cortex Search service and Semantic View created in previous notebooks.

### 5.1: Verify Existing Tools

First, let's verify that the tools exist and are accessible.

In [ ]:
USE ROLE ACCOUNTADMIN;

-- Verify Cortex Search Service exists
DESCRIBE CORTEX SEARCH SERVICE IDENTIFIER($SEARCH_SERVICE_FULL_PATH);

-- Verify Semantic View exists
DESCRIBE SEMANTIC VIEW IDENTIFIER($SEMANTIC_VIEW_FULL_PATH);

### 5.2: Grant Access to Cortex Search Service

Grant USAGE privilege on the existing Cortex Search service to agent roles.

In [ ]:
USE ROLE ACCOUNTADMIN;

-- Grant database and schema access for the search service
GRANT USAGE ON DATABASE RUNBOOKS_DB TO ROLE IDENTIFIER($AGENT_CREATOR_ROLE);
GRANT USAGE ON SCHEMA RUNBOOKS_DB.CORTEX_SEARCH TO ROLE IDENTIFIER($AGENT_CREATOR_ROLE);

-- Grant USAGE on the Cortex Search Service
GRANT USAGE ON CORTEX SEARCH SERVICE IDENTIFIER($SEARCH_SERVICE_FULL_PATH) 
    TO ROLE IDENTIFIER($AGENT_CREATOR_ROLE);

### 5.3: Grant Access to Semantic View

Grant SELECT and REFERENCES privileges on the existing semantic view to agent roles.

In [ ]:
USE ROLE ACCOUNTADMIN;

-- Grant database and schema access for the semantic view
GRANT USAGE ON SCHEMA RUNBOOKS_DB.CORTEX_ANALYST TO ROLE IDENTIFIER($AGENT_CREATOR_ROLE);

-- Grant SELECT on the Semantic View (required for Cortex Analyst)
GRANT SELECT ON SEMANTIC VIEW IDENTIFIER($SEMANTIC_VIEW_FULL_PATH) 
    TO ROLE IDENTIFIER($AGENT_CREATOR_ROLE);


-- Grant REFERENCES if needed (for some Cortex Analyst operations)
GRANT REFERENCES ON SEMANTIC VIEW IDENTIFIER($SEMANTIC_VIEW_FULL_PATH) 
    TO ROLE IDENTIFIER($AGENT_CREATOR_ROLE);

## Part 6: Create Cortex Agent

Create an agent that uses both **existing** Cortex Analyst (for sales data) and Cortex Search (for product documentation).

**NOTE**: creating agents doesn't work with session variables in the specification. Hence, the path's to the tools are hardcoded.

In [ ]:
USE ROLE IDENTIFIER($AGENT_CREATOR_ROLE);
USE SCHEMA IDENTIFIER($FULL_SCHEMA_PATH);

CREATE OR REPLACE AGENT IDENTIFIER($AGENT_NAME)
    COMMENT = 'Sales analytics agent using existing tools from previous notebooks'
    PROFILE = '{"display_name": "Sales Analytics Assistant"}'
    FROM SPECIFICATION $$
    {
        "models": {
            "orchestration": "claude-4-sonnet"
        },
        "instructions": {
            "orchestration": "Use sales_analyst for querying sales data and metrics from the SALES_SEMANTIC_VIEW_RESTRICTED semantic view. Use product_search for product documentation from PRODUCT_SEARCH_SERVICE. Always provide data-driven insights.",
            "response": "Be concise, professional, and data-focused. Include relevant metrics when discussing sales performance."
        },
        "tools": [
            {
                "tool_spec": {
                    "type": "cortex_analyst_text_to_sql",
                    "name": "sales_analyst",
                    "description": "Query sales data including revenue, orders, products, and regional performance"
                }
            },
            {
                "tool_spec": {
                    "type": "cortex_search",
                    "name": "product_search",
                    "description": "Search product documentation, specifications, and support information"
                }
            }
        ],
        "tool_resources": {
            "sales_analyst": {
                "semantic_view": "RUNBOOKS_DB.CORTEX_ANALYST.SALES_SEMANTIC_VIEW_RESTRICTED",
                "execution_environment": {
                    "type": "warehouse",
                    "warehouse": "COMPUTE_WH"
                },
                "query_timeout": 60
            },
            "product_search": {
                "search_service": "RUNBOOKS_DB.CORTEX_SEARCH.PRODUCT_SEARCH_SERVICE",
                "max_results": 10,
                "columns": ["product_name", "category", "documentation"]
            }
        }
    }
    $$;

**Note**: The agent references fully-qualified paths to existing tools from previous notebooks.

## Part 7: Grant Agent-Level Privileges

Grant specific privileges on the agent to different roles.

### 7.1: Grant USAGE Privilege to Agent Users

In [ ]:
USE ROLE ACCOUNTADMIN;
USE SCHEMA IDENTIFIER($FULL_SCHEMA_PATH);

GRANT USAGE ON DATABASE IDENTIFIER($DATABASE_NAME) TO ROLE IDENTIFIER($AGENT_USER_ROLE);
GRANT USAGE ON SCHEMA IDENTIFIER($FULL_SCHEMA_PATH) TO ROLE IDENTIFIER($AGENT_USER_ROLE);

GRANT USAGE ON AGENT IDENTIFIER($AGENT_NAME) 
    TO ROLE IDENTIFIER($AGENT_USER_ROLE);

SHOW GRANTS TO ROLE IDENTIFIER($AGENT_USER_ROLE);

### 7.2: Grant MODIFY Privilege to Operators

In [ ]:
USE ROLE ACCOUNTADMIN;
USE SCHEMA IDENTIFIER($FULL_SCHEMA_PATH);

GRANT USAGE ON DATABASE IDENTIFIER($DATABASE_NAME) TO ROLE IDENTIFIER($AGENT_OPERATOR_ROLE);
GRANT USAGE ON SCHEMA IDENTIFIER($FULL_SCHEMA_PATH) TO ROLE IDENTIFIER($AGENT_OPERATOR_ROLE);

GRANT MODIFY ON AGENT IDENTIFIER($AGENT_NAME) 
    TO ROLE IDENTIFIER($AGENT_OPERATOR_ROLE);

## Part 8: Cleanup

Remove all created objects (use with caution in production environments).

**Note**: This cleanup ONLY removes objects created in this notebook. It does NOT remove tools from previous notebooks.

In [ ]:
USE ROLE ACCOUNTADMIN;

-- Drop the agent
--DROP AGENT IF EXISTS IDENTIFIER($AGENT_NAME);

-- Drop warehouse
--DROP WAREHOUSE IF EXISTS IDENTIFIER($WAREHOUSE_NAME);

-- Drop roles
--DROP ROLE IF EXISTS IDENTIFIER($AGENT_CREATOR_ROLE);
--DROP ROLE IF EXISTS IDENTIFIER($AGENT_USER_ROLE);
--DROP ROLE IF EXISTS IDENTIFIER($AGENT_OPERATOR_ROLE);

-- Drop schema and database (commented out for safety)
-- DROP SCHEMA IF EXISTS IDENTIFIER($FULL_SCHEMA_PATH);
-- DROP DATABASE IF EXISTS IDENTIFIER($DATABASE_NAME);

-- NOTE: We DO NOT drop the Cortex Search service or Semantic View
-- as they belong to previous notebooks and may still be in use

---

## Additional Resources

- [Cortex Agents Documentation](https://docs.snowflake.com/user-guide/snowflake-cortex/cortex-agents)
- [Cortex Agents REST API](https://docs.snowflake.com/user-guide/snowflake-cortex/cortex-agents-rest-api)
- [Snowflake Intelligence Overview](https://docs.snowflake.com/user-guide/snowflake-cortex/snowflake-intelligence)
- [Access Control Privileges](https://docs.snowflake.com/user-guide/security-access-control-privileges)

---

**Version:** 2.0  
**Last Updated:** March 2026